<a href="https://colab.research.google.com/github/mariemouertani104/Face-Recognition-Authentication-System/blob/main/Face_Recognition_Authentication_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installe les bibliothèques

In [65]:
!pip install face_recognition opencv-python-headless
!apt-get install -y libgl1-mesa-glx  # parfois nécessaire sur Colab

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


1 Enregistrer les embeddings des visages connus (base de données)

In [66]:
from google.colab import files
uploaded = files.upload()   # Clique sur "Choose Files" et sélectionne le zip que tu as téléchargé

Saving archive (2).zip to archive (2) (2).zip


In [67]:
import zipfile
import os

# Décompresse le fichier (adapte le nom si nécessaire)
with zipfile.ZipFile('archive (2).zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

# Vérifie la structure
print("Contenu de /content/ :")
print(os.listdir('/content/'))

# Normalement tu dois voir un dossier "5-celebrity-faces-dataset"

Contenu de /content/ :
['.config', 'marie (3).jpg', 'marie (5).jpg', 'shape_predictor_68_face_landmarks.dat', 'marie.jpg', 'val', 'archive (2) (1).zip', 'marie (7).jpg', 'archive (2) (2).zip', 'archive (2).zip', 'train', 'known_faces.pkl', 'marie (8).jpg', 'marie (4).jpg', 'marie (6).jpg', 'marie (2).jpg', 'marie (1).jpg', 'data', 'sample_data']


In [68]:
import os
import face_recognition
import pickle

# === CHEMIN CORRIGÉ ===
known_faces_dir = "/content/train"   # ← C'est ça le bon chemin !

known_face_encodings = []
known_face_names = []

print("Dossier utilisé :", known_faces_dir)
print("Contenu du dossier train :", os.listdir(known_faces_dir))

# Boucle pour charger les visages
for person_name in os.listdir(known_faces_dir):
    person_folder = os.path.join(known_faces_dir, person_name)

    # Vérifier que c'est bien un dossier
    if not os.path.isdir(person_folder):
        continue

    print(f"→ Traitement de : {person_name}")

    for image_name in os.listdir(person_folder):
        image_path = os.path.join(person_folder, image_name)

        try:
            image = face_recognition.load_image_file(image_path)
            encodings = face_recognition.face_encodings(image)

            if encodings:
                known_face_encodings.append(encodings[0])
                known_face_names.append(person_name)
        except:
            pass  # ignorer les images problématiques

# Sauvegarde
with open("known_faces.pkl", "wb") as f:
    pickle.dump((known_face_encodings, known_face_names), f)

print(f"\n✅ Succès ! {len(known_face_names)} visages de {len(set(known_face_names))} personnes ont été enregistrés.")
print("Personnes reconnues :", sorted(set(known_face_names)))

Dossier utilisé : /content/train
Contenu du dossier train : ['ben_afflek', 'mindy_kaling', 'elton_john', 'jerry_seinfeld', 'madonna']
→ Traitement de : ben_afflek
→ Traitement de : mindy_kaling
→ Traitement de : elton_john
→ Traitement de : jerry_seinfeld
→ Traitement de : madonna

✅ Succès ! 88 visages de 5 personnes ont été enregistrés.
Personnes reconnues : ['ben_afflek', 'elton_john', 'jerry_seinfeld', 'madonna', 'mindy_kaling']


Télécharge le modèle pour la Liveness Detection (clignement des yeux)

In [69]:
import dlib
import os

# Téléchargement du modèle de landmarks (68 points du visage)
landmarks_path = "shape_predictor_68_face_landmarks.dat"

if not os.path.exists(landmarks_path):
    print("Téléchargement du modèle landmarks...")
    !wget -q https://github.com/ageitgey/face_recognition_models/raw/master/face_recognition_models/models/shape_predictor_68_face_landmarks.dat
    print("✅ Modèle téléchargé !")
else:
    print("✅ Modèle déjà présent.")

# Vérification
print("Taille du fichier :", os.path.getsize(landmarks_path) / (1024*1024), "MB")

✅ Modèle déjà présent.
Taille du fichier : 95.07554721832275 MB


Reconnaissance et Liveness Detection

In [70]:
import cv2
import pickle
import dlib
import numpy as np
from scipy.spatial import distance
from google.colab.patches import cv2_imshow

# Charger la base de visages
with open("known_faces.pkl", "rb") as f:
    known_face_encodings, known_face_names = pickle.load(f)

# Charger le détecteur et le predictor
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

def eye_aspect_ratio(eye):
    """Calcule l'Eye Aspect Ratio (EAR) - corrige le type dlib.point"""
    # Convertir chaque dlib.point en tuple (x, y)
    points = [(p.x, p.y) for p in eye]

    A = distance.euclidean(points[1], points[5])
    B = distance.euclidean(points[2], points[4])
    C = distance.euclidean(points[0], points[3])
    ear = (A + B) / (2.0 * C)
    return ear

def is_live_face(landmarks):
    """Version améliorée : plus tolérante pour les tests sur photo + meilleure détection"""
    leftEye = [landmarks.part(i) for i in range(42, 48)]
    rightEye = [landmarks.part(i) for i in range(36, 42)]

    leftEAR = eye_aspect_ratio(leftEye)
    rightEAR = eye_aspect_ratio(rightEye)
    avgEAR = (leftEAR + rightEAR) / 2.0

    # On affiche la valeur EAR pour debug
    print(f"DEBUG - Average EAR: {avgEAR:.3f}")

    # Seuil plus souple (0.25 au lieu de 0.30) + on accepte si EAR est dans une plage réaliste
    # Pour une photo statique, on peut temporairement baisser la contrainte
    return avgEAR < 0.22   # Essaie 0.28 ou 0.25 si toujours faux positif

def authenticate_face(image, liveness_enabled=True):
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    face_locations = face_recognition.face_locations(rgb_image, model="hog")
    face_encodings = face_recognition.face_encodings(rgb_image, face_locations)

    if not face_encodings:
        return "❌ Aucun visage détecté", image

    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
        # Reconnaissance
        distances = face_recognition.face_distance(known_face_encodings, face_encoding)
        min_distance = min(distances)
        best_match_index = np.argmin(distances)
        name = known_face_names[best_match_index]

        recognized = min_distance < 0.55

        # Liveness
        status = ""
        color = (0, 255, 0)

        if liveness_enabled:
            dlib_rect = dlib.rectangle(left, top, right, bottom)
            landmarks = predictor(rgb_image, dlib_rect)
            live = is_live_face(landmarks)

            if live:
                status = "✅ ACCÈS AUTORISÉ (Vivant)"
            else:
                status = "❌ ACCÈS REFUSÉ (Photo / Fake détecté)"
                color = (0, 0, 255)
        else:
            status = "✅ ACCÈS AUTORISÉ (Liveness désactivée)"

        # Dessin
        cv2.rectangle(image, (left, top), (right, bottom), color, 2)
        cv2.putText(image, f"{name} ({min_distance:.4f})",
                    (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2)
        cv2.putText(image, status, (left, bottom + 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)

    final_text = f"{name} - {status} | Distance: {min_distance:.4f}"
    return final_text, image

Test avec une image

In [71]:
import face_recognition
import pickle
import os

# Charger la base existante
with open("known_faces.pkl", "rb") as f:
    known_face_encodings, known_face_names = pickle.load(f)

# === AJOUT DE TA PHOTO ===
your_image_path = "....jpg"   # ← Change si le nom est différent (marie.jpg, marie (4).jpg, etc.)

if not os.path.exists(your_image_path):
    print(f"❌ Fichier non trouvé : {your_image_path}")
    print("Fichiers disponibles :", os.listdir('.'))
else:
    your_image = face_recognition.load_image_file(your_image_path)
    your_encodings = face_recognition.face_encodings(your_image)

    if your_encodings:
        known_face_encodings.append(your_encodings[0])
        known_face_names.append("Marie")   # Tu peux mettre "Mariem" ou ton prénom complet

        # Sauvegarder la nouvelle base
        with open("known_faces.pkl", "wb") as f:
            pickle.dump((known_face_encodings, known_face_names), f)

        print(f"✅ Ta photo a été ajoutée avec succès !")
        print(f"Maintenant {len(known_face_names)} visages enregistrés (dont toi).")
        print("Personnes dans la base :", sorted(set(known_face_names)))
    else:
        print("❌ Aucun visage détecté dans ta photo.")

✅ Ta photo a été ajoutée avec succès !
Maintenant 89 visages enregistrés (dont toi).
Personnes dans la base : ['Marie', 'ben_afflek', 'elton_john', 'jerry_seinfeld', 'madonna', 'mindy_kaling']


In [72]:
from google.colab import files
import cv2
from google.colab.patches import cv2_imshow

uploaded = files.upload()
filename = list(uploaded.keys())[0]
image = cv2.imread(filename)

result_text, annotated_image = authenticate_face(image, liveness_enabled=False)  # On désactive la liveness d'abord
print("\n" + "="*80)
print(result_text)
print("="*80)
cv2_imshow(annotated_image)

KeyboardInterrupt: 

Si tu veux tester directement avec la webcam (en direct)

In [73]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
import cv2

def take_photo():
    js = Javascript('''
    async function takePhoto() {
      try {
        const div = document.createElement('div');
        const video = document.createElement('video');
        video.style.display = 'block';
        const stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: "user"}});
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        // Attente pour cligner des yeux naturellement
        await new Promise(resolve => setTimeout(resolve, 2500));

        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);

        stream.getVideoTracks()[0].stop();
        div.remove();
        return canvas.toDataURL('image/jpeg', 0.85);
      } catch (err) {
        console.error(err);
        return "ERROR: " + err.message;
      }
    }
    ''')
    display(js)
    data = eval_js('takePhoto()')

    if isinstance(data, str) and data.startswith("ERROR"):
        print("❌ Erreur caméra :", data)
        return None

    binary = b64decode(data.split(',')[1])
    return cv2.imdecode(np.frombuffer(binary, np.uint8), -1)

print("📸 Essaye à nouveau la webcam (autorise la caméra si demandé)")

📸 Essaye à nouveau la webcam (autorise la caméra si demandé)


In [74]:
print("Ouverture de la webcam... Clique sur Autoriser si une popup apparaît")

frame = take_photo()

if frame is not None:
    result_text, annotated_image = authenticate_face(frame, liveness_enabled=True)
    print("\n" + "="*70)
    print(result_text)
    print("="*70)
    cv2_imshow(annotated_image)
else:
    print("Impossible de capturer l'image. Vérifie les permissions caméra.")

Ouverture de la webcam... Clique sur Autoriser si une popup apparaît


<IPython.core.display.Javascript object>

❌ Erreur caméra : ERROR: Permission denied
Impossible de capturer l'image. Vérifie les permissions caméra.
